# Data Analysis & Modeling: Predict Whether a Player Joins an Experience

**Original interview mock, not a reported Roblox question or an official CodeSignal assessment.**
Interview format: **5 minutes introduction + 35 minutes coding + 5 minutes Q&A**. All data are synthetic.

## The interviewer prompt

At the time an experience card is shown, predict whether the player joins that experience within the next ten minutes. Each row is one displayed card (an impression), not one player.

**Problem 1 (14 minutes):** inspect and clean impression logs, attach experience metadata, and report join rate by genre.
**Problem 2 (21 minutes):** build a probability model, use a user-disjoint holdout, evaluate imbalanced classification, and score new cards.

This is a small pointwise recommendation baseline, not a full recommendation system. Ranking cards is an optional follow-up after the timed exercise.

| Coding time | Task |
| --- | --- |
| 0–4 min | Load and inspect |
| 4–9 min | Clean the data |
| 9–14 min | Analyze and prepare features |
| 14–18 min | Select features and split |
| 18–27 min | Preprocess and train |
| 27–32 min | Evaluate |
| 32–35 min | Predict and summarize |

These are suggested checkpoints, not official sub-round timings. Ask during the introduction: What is one example? When is a prediction made? What exactly is the target?

**Practice version.** Write your own imports and setup, then complete the seven exercise steps. Input CSVs and per-step checks are provided. Replace each NotImplementedError with your implementation. Use the matching solution notebook when reviewing.

## Data contract — supplied by the interviewer

The interviewer supplies **impressions.csv** and **experiences.csv** in **data/codesignal_roblox_recommendation/**. Load these into DataFrames named **impressions** and **experiences** in Step 1. Each user has one request with four shown candidates. All ten-minute outcomes are observed. Historical affinity was computed before the request; experience attributes are snapshots available at request time.

| Column | Meaning / policy |
| --- | --- |
| impression_id | Unique displayed-card ID, except exact repeated exports |
| user_id, request_id, experience_id | Group/join identifiers, not numeric features |
| device | Device at request time, may be missing |
| prior_genre_minutes | User's minutes in that genre during the previous seven days; negatives mean unknown |
| friends_playing | Number of friends playing at request time |
| position | Display position 1–4; affects observed behavior |
| joined | 0/1 label: join this experience within ten minutes |
| minutes_after_impression | Subsequent play time; unavailable for the current prediction |
| experiences.genre, experience_age_days | Static/as-of-request experience metadata |

The evaluation question is explicitly **generalization to unseen users**. Users must not cross the train/test boundary. Multiple joins per request are possible over ten minutes; this is not a mutually exclusive softmax task. Logs contain only displayed candidates, not all possible experiences.


## Your setup — write the imports

Open this notebook from the workspace root in VS Code or Jupyter and select **.venv-codesignal/bin/python**. No network, GPU or additional package installation is required.

The two input files are **data/codesignal_roblox_recommendation/impressions.csv** and **data/codesignal_roblox_recommendation/experiences.csv**. Load them into impressions and experiences in Step 1. The source files are already supplied; data generation is not part of the exercise.

Write the imports you need in the next cell. Define **SEED = 42** and **DATA_DIR** for the CSV directory. You may use a pathlib directory or ordinary string paths. Restart the kernel before attempting the exercise so imports from other notebooks do not hide missing imports. If doing a timed mock, include writing imports and setup in your 35 minutes.


In [55]:
# Write your imports here.
# Define SEED as 42 and DATA_DIR for the supplied CSV directory.
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn import metrics

SEED = 42
#raise NotImplementedError("Complete setup, then start Step 1.")


## Problem 1 · Step 1 — Load and inspect (0–4 min)

Load impressions.csv and experiences.csv into impressions and experiences with pd.read_csv. Inspect both tables. Count users, requests, duplicate impressions, missingness and joined class balance.

**Say before coding:** "One example is a displayed user-experience pair. Repeated users make a naive random row split risky for our unseen-user evaluation."

**Possible interviewer questions**

1. What does one training example represent?
2. Are unshown items negative examples?
3. What should you inspect before dropping missing rows?


In [22]:
# Create the variables named in the task above.
#
IMPRESSION_PATH = "./data/codesignal_roblox_recommendation/impressions.csv"
EXPERIENCE_PATH = "./data/codesignal_roblox_recommendation/experiences.csv"

impressions = pd.read_csv(IMPRESSION_PATH)
experiences = pd.read_csv(EXPERIENCE_PATH)

#display(impressions.info())
display(impressions.head())
#display(experiences.info())
display(experiences.head())

print("user count", impressions["user_id"].nunique())
print("request count", impressions["request_id"].nunique())


print("joined propations", impressions['joined'].value_counts(normalize=True))
# 
# raise NotImplementedError("Complete step 1, then run the checks below.")


,impression_id,user_id,request_id,experience_id,device,position,prior_genre_minutes,friends_playing,joined,minutes_after_impression
0,0,0,0,51,desktop,1,27.028748,1,1,20.215565
1,1,0,0,37,desktop,2,47.208345,1,1,13.380891
2,2,0,0,38,desktop,3,26.547296,1,0,0.000000
3,3,0,0,27,desktop,4,17.268351,0,0,0.000000
4,4,1,1,41,mobile,1,23.604279,1,0,0.000000


,experience_id,genre,experience_age_days
0,0,adventure,1383
1,1,puzzle,1117
2,2,simulation,550
3,3,simulation,1451
4,4,simulation,616


user count 600
request count 600
joined propations joined
0    0.918629
1    0.081371
Name: proportion, dtype: float64


In [21]:
assert experiences["experience_id"].is_unique
assert impressions["experience_id"].isin(experiences["experience_id"]).all()
assert set(impressions["joined"]) == {0, 1}


## Problem 1 · Step 2 — Clean the data (4–9 min)

Create clean: remove exact duplicates and turn negative prior_genre_minutes into NaN. Join experiences into frame using a left many-to-one merge. Do not aggregate impressions away.

**Say before coding:** "A popular experience can appear many times, but its metadata must match only one row per impression."

**Possible interviewer questions**

1. Why not one_to_one?
2. Can we use minutes_after_impression?
3. Why postpone median imputation until after splitting?


In [26]:
# Create the variables named in the task above.
##
cleaned_impressions = impressions.drop_duplicates().copy()
mask = cleaned_impressions["prior_genre_minutes"].lt(0)
cleaned_impressions.loc[mask,"prior_genre_minutes"] = np.nan



frame = cleaned_impressions.merge(experiences, on="experience_id", how="left", validate="many_to_one")
display(frame.head())

print(len(frame))
print(impressions["impression_id"].nunique())

mask = frame["impression_id"].duplicated(keep=False)

conflicts = frame.loc[mask]
frame = frame.loc[~mask].copy()

print(len(frame))



#raise NotImplementedError("Complete step 2, then run the checks below.")


,impression_id,user_id,request_id,experience_id,device,position,prior_genre_minutes,friends_playing,joined,minutes_after_impression,genre,experience_age_days
0,0,0,0,51,desktop,1,27.028748,1,1,20.215565,adventure,1171
1,1,0,0,37,desktop,2,47.208345,1,1,13.380891,puzzle,1024
2,2,0,0,38,desktop,3,26.547296,1,0,0.000000,adventure,955
3,3,0,0,27,desktop,4,17.268351,0,0,0.000000,puzzle,469
4,4,1,1,41,mobile,1,23.604279,1,0,0.000000,puzzle,300


2401
2400
2399


In [ ]:
assert len(frame) == impressions["impression_id"].nunique()
assert frame["impression_id"].is_unique
assert frame["prior_genre_minutes"].dropna().ge(0).all()


## Problem 1 · Step 3 — Analyze and prepare features (9–14 min)

Create genre_summary with impression_count, joins and join_rate, grouped by genre. Also report join_rate by position. Explain why observed rates are not causal effects.

**Say before coding:** "The mean of a 0/1 outcome is its observed rate. Group counts tell us how much evidence each rate uses."

**Possible interviewer questions**

1. Why include counts next to rates?
2. Does higher join rate prove a genre is better?
3. How do you check that a merge did not multiply rows?


In [31]:
# Create the variables named in the task above.
genere_summary = frame.groupby("genre").agg(
    impression_count = ("impression_id", "count"),
    joins = ("joined","sum"),
    join_rate = ("joined", "mean"),
)

display(genere_summary.head().T)

join_by_position = frame.groupby("position").agg(
    join_rate = ("joined", "mean"),
)
display(join_by_position.head().T)


##raise NotImplementedError("Complete step 3, then run the checks below.")


genre,adventure,puzzle,simulation
impression_count,605.000000,946.000000,848.000000
joins,54.000000,83.000000,55.000000
join_rate,0.089256,0.087738,0.064858


position,1,2,3,4
join_rate,0.113523,0.095,0.058333,0.053333


In [ ]:
assert genre_summary["impression_count"].sum() == len(frame)
np.testing.assert_allclose(genre_summary["join_rate"], genre_summary["joins"] / genre_summary["impression_count"])


## Problem 2 · Step 4 — Select features and split (14–18 min)

Which columns would you use as features, which column is the target, and which fields would you exclude? We want to evaluate how well the model predicts joins for users it has never seen during training. What kind of train/test split would you choose, and why? Implement your choice and explain how you would verify that the split matches this evaluation goal.

**Say before coding:** "I will clarify the evaluation goal, choose a splitting strategy, and check that the resulting sets support that goal."

**Possible interviewer questions**

1. Would a random split of impression rows answer this evaluation question?
2. What would you check after splitting?
3. How do you check that X and y still correspond?


In [38]:
# Create the variables named in the task above.
numeric_features = ["prior_genre_minutes","friends_playing","position","experience_age_days"]
categorical_features = ["device", "genre"]

X = frame[numeric_features +categorical_features ]
y = frame["joined"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)

train_idx, test_idx = next(splitter.split(X,y,groups=frame["user_id"]))
print("train_idx", train_idx)
print("test_idx", test_idx)
print(X.head())
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]



#raise NotImplementedError("Complete step 4, then run the checks below.")


train_idx [   0    1    2 ... 2396 2397 2398]
test_idx [   8    9   10   11   24   25   26   27   40   41   42   43   44   45
   46   47   96   97   98   99  115  116  117  118  119  120  121  122
  155  156  157  158  219  220  221  222  239  240  241  242  251  252
  253  254  279  280  281  282  287  288  289  290  291  292  293  294
  303  304  305  306  307  308  309  310  311  312  313  314  315  316
  317  318  323  324  325  326  327  328  329  330  331  332  333  334
  343  344  345  346  359  360  361  362  403  404  405  406  415  416
  417  418  435  436  437  438  439  440  441  442  471  472  473  474
  527  528  529  530  539  540  541  542  547  548  549  550  559  560
  561  562  591  592  593  594  619  620  621  622  651  652  653  654
  659  660  661  662  703  704  705  706  723  724  725  726  727  728
  729  730  735  736  737  738  767  768  769  770  779  780  781  782
  795  796  797  798  831  832  833  834  835  836  837  838  839  840
  841  842  847  848  

In [ ]:
assert set(frame.iloc[train_idx]["user_id"]).isdisjoint(frame.iloc[test_idx]["user_id"])
assert X_train.index.equals(y_train.index) and X_test.index.equals(y_test.index)
assert y_train.nunique() == y_test.nunique() == 2
assert not {"user_id", "request_id", "experience_id", "impression_id", "joined", "minutes_after_impression"} & set(X.columns)


## Problem 2 · Step 5 — Preprocess and train (18–27 min)

Create numeric_pipeline with median imputation and StandardScaler, and categorical_pipeline with most-frequent imputation and OneHotEncoder(handle_unknown="ignore"). Combine them in preprocess using ColumnTransformer. Create model as a Pipeline with preprocess and LogisticRegression(max_iter=500), and baseline as DummyClassifier(strategy="prior"). Fit both using training data only.

**Say before coding:** "I will start with an interpretable pointwise probability model, not a complex recommender."

**Possible interviewer questions**

1. Should class_weight="balanced" always be on?
2. Why keep position for the classification exercise?
3. What happens when you call model.fit on the Pipeline?


In [45]:
# Create the variables named in the task above.
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy='median')),
    ("normalize", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy='most_frequent')),
    ("encode", OneHotEncoder()),
])

preprocess = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

model = Pipeline([
    ("preprocess", preprocess),
    ("classifier", LogisticRegression(max_iter=500,verbose=2)),
])

model.fit(X_train, y_train)

baseline = DummyClassifier(strategy="prior").fit(X_train, y_train)








#raise NotImplementedError("Complete step 5, then run the checks below.")


RUNNING THE L-BFGS-B CODE

           * * *

Machine precision = 2.220D-16
 N =           11     M =           10

At X0         0 variables are exactly at the bounds

At iterate    0    f=  6.93147D-01    |proj g|=  4.23398D-01

At iterate    1    f=  3.36238D-01    |proj g|=  1.40527D-01

At iterate    2    f=  2.70271D-01    |proj g|=  5.33458D-02

At iterate    3    f=  2.50641D-01    |proj g|=  1.90904D-02

At iterate    4    f=  2.45382D-01    |proj g|=  7.01763D-03

At iterate    5    f=  2.44544D-01    |proj g|=  2.78705D-03

At iterate    6    f=  2.44454D-01    |proj g|=  2.25294D-03

At iterate    7    f=  2.44423D-01    |proj g|=  8.48773D-04

At iterate    8    f=  2.44407D-01    |proj g|=  5.94363D-04

At iterate    9    f=  2.44396D-01    |proj g|=  4.35041D-04

At iterate   10    f=  2.44382D-01    |proj g|=  3.27594D-04

At iterate   11    f=  2.44328D-01    |proj g|=  1.00284D-03

At iterate   12    f=  2.44245D-01    |proj g|=  1.49691D-03

At iterate   13    f=  2.4

 This problem is unconstrained.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s finished



At iterate   17    f=  2.43950D-01    |proj g|=  5.90874D-05

           * * *

Tit   = total number of iterations
Tnf   = total number of function evaluations
Tnint = total number of segments explored during Cauchy searches
Skip  = number of BFGS updates skipped
Nact  = number of active bounds at final generalized Cauchy point
Projg = norm of the final projected gradient
F     = final function value

           * * *

   N    Tit     Tnf  Tnint  Skip  Nact     Projg        F
   11     17     18      1     0     0   5.909D-05   2.439D-01
  F =  0.24394988075784205     

CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL            


In [ ]:
np.testing.assert_array_equal(model.classes_, [0, 1])
learned = model.named_steps["preprocess"].named_transformers_["num"].named_steps["imputer"].statistics_
np.testing.assert_allclose(learned, X_train[numeric_features].median().to_numpy())


## Problem 2 · Step 6 — Evaluate (27–32 min)

Create prob=model.predict_proba(X_test)[:,1] and pred at threshold 0.5. Compare baseline and model on ROC-AUC, average precision, log loss, precision and recall at 0.5. Show the confusion matrix.

**Say before coding:** "A model can rank useful candidates well while producing few positive decisions at the default threshold."

**Possible interviewer questions**

1. Why might accuracy look good for a useless model?
2. Where should we choose a different threshold?
3. Can you keep changing the model after looking at test metrics?


In [57]:
# Create the variables named in the task above.
prob = model.predict_proba(X_test)[:,1]
#print("prob",prob)


def evaluate(pred_prob, gt):
    pred = (pred_prob >= 0.5).astype(int)
    return {
        "roc_auc": metrics.roc_auc_score(gt,pred_prob),
        "average_precision": metrics.average_precision_score(gt,pred_prob),
        "log_loss": metrics.log_loss(gt,pred_prob),
        "precision@0.5": metrics.precision_score(gt,pred),
        "recall@0.5": metrics.recall_score(gt,pred)
    }


print(evaluate(prob,y_test))



#raise NotImplementedError("Complete step 6, then run the checks below.")


{'roc_auc': 0.5555555555555556, 'average_precision': 0.19444444444444442, 'log_loss': 0.28882924895981266, 'precision@0.5': 1.0, 'recall@0.5': 0.1111111111111111}


In [ ]:
assert prob.shape == (len(y_test),)
assert ((prob >= 0) & (prob <= 1)).all()
np.testing.assert_allclose(scores.loc["prior", "average_precision"], y_test.mean())
np.testing.assert_allclose(scores.loc["prior", "roc_auc"], 0.5)


## Problem 2 · Step 7 — Predict and summarize (32–35 min)

Create new_cards with two rows containing all input features. Use the same assumed position=1 for both candidate cards and compute join_prob. Explain why score-based ranking still needs a separate evaluation.

**Say before coding:** "For comparing candidates I hold position constant, but these observational probabilities are not causal estimates."

**Possible interviewer questions**

1. Is global ROC-AUC the same as ranking quality per request?
2. Can this model score every unseen experience?
3. What must training and inference share?


In [ ]:
# Create the variables named in the task above.
raise NotImplementedError("Complete step 7, then run the checks below.")


In [ ]:
assert join_prob.shape == (2,)
assert np.isfinite(join_prob).all()


## If the interviewer extends the problem

**Optional ranking follow-up (not in the 35-minute core):** score held-out requests with position held at 1, sort each request by score, and compute Recall@2 over requests with at least one observed join. State that these are outcomes under the old display policy, not unbiased counterfactual labels for a new ordering. Report how many requests were excluded because their denominator is zero. This diagnostic is not proof that re-ranking will improve joins.


## Scope and sources

These are **original synthetic exercises**, not reported Roblox questions and not official CodeSignal assessments. The user-supplied **Data Analysis & Modeling Coding Interview** guide defines the format: 5 minutes intro, 35 minutes for two problems (analysis and modeling), 5 minutes Q&A; pandas, NumPy, sklearn and SciPy are available. The per-step times here are suggested practice checkpoints, not official grading criteria.

The guide does not identify a team or prescribe regression/softmax topics. We chose those to practice task-appropriate modeling and metrics. Business themes are informed by the [official PhD Early Career MLE posting](https://careers.roblox.com/jobs/8242623), which mentions engagement, safety and recommendation systems. Source checked 2026-10-22; exact interview questions are unknown.

Implementation references: [sklearn 1.6 LogisticRegression](https://scikit-learn.org/1.6/modules/generated/sklearn.linear_model.LogisticRegression.html), [model evaluation](https://scikit-learn.org/1.6/modules/model_evaluation.html), [data leakage](https://scikit-learn.org/1.6/common_pitfalls.html#data-leakage).
